In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

In [ ]:
def jacobi_preconditioned_conjugate_gradient(A: np.ndarray, b: np.ndarray, x0: np.ndarray = None, tol: float = 1e-10, max_iter: int = 1000) -> np.ndarray:
    """
    Solve the linear system Ax = b using the Jacobi Preconditioned Conjugate Gradient method.
    Parameters:
    A : np.ndarray
        Symmetric positive-definite matrix.
    b : np.ndarray
        Right-hand side vector.
    x0 : np.ndarray, optional
        Initial guess for the solution (default is a zero vector).
    tol : float, optional
        Tolerance for convergence (default is 1e-10).
    max_iter : int, optional
        Maximum number of iterations (default is 1000).
    """

    # number of rows
    n = A.shape[0]

    if x0 is None:
        x = np.zeros(n)
    else:
        x = x0.copy()

    r = b - A @ x
    M_inv = 1 / np.diag(A)  # Jacobi preconditioner (inverse of diagonal elements)
    z = M_inv * r
    p = z.copy()
    rs_old = np.dot(r, z)

    for i in range(max_iter):
        Ap = A @ p
        alpha = rs_old / np.dot(p, Ap)

        x = x + alpha * p
        r = r - alpha * Ap
        
        z = M_inv * r
        rs_new = np.dot(r, z)
        if np.sqrt(rs_new) / np.sqrt(rs_old) < tol:
            print("Converged")
            break

        beta = rs_new / rs_old

        p = z + beta * p

        rs_old = rs_new

    return x   

# Example usage
if __name__ == "__main__":
    # Define a symmetric positive-definite matrix A and vector b
    A = np.array([[4, 1], [1, 3]], dtype=float)
    b = np.array([1, 2], dtype=float)

    # Solve for x in Ax = b
    import time
    start_time = time.time()
    x = jacobi_preconditioned_conjugate_gradient(A, b)
    end_time = time.time()
    print("Time taken:", (end_time - start_time) / 60, "minutes")
    assert np.allclose(x, np.array([0.090909, 0.636364]))
    print("Solution x:", x)

In [ ]:
def check_spd_properties(A: np.ndarray) -> None:
    print(f"Matrix shape: {A.shape}")
    print(f"Matrix:\n{A}\n")

    # Check symmetry
    is_sym = np.allclose(A, A.T, atol=1e-10)
    print(f"✓ Symmetric: {is_sym}")

    # Check SPD via eigenvalues
    eigvals = np.linalg.eigvalsh(A)
    print(f"Eigenvalues: {eigvals}")
    is_spd = np.all(eigvals > 1e-10)
    print(f"✓ Positive definite (all eigenvalues > 0): {is_spd}")

    # Condition number
    if is_spd:
        cond_num = np.max(eigvals) / np.min(eigvals)
        print(f"Condition number: {cond_num:.2e}")

    # Check diagonal dominance
    diag = np.diag(A)
    row_sums = np.sum(np.abs(A), axis=1) - np.abs(diag)
    is_dd = np.all(np.abs(diag) > row_sums)
    print(f"✓ Diagonally dominant: {is_dd}")

    # Check 27-point stencil structure (for 3D Laplacian on a grid)
    n = A.shape[0]
    grid_size = round(n ** (1/3))
    
    if grid_size ** 3 == n:
        print(f"\n✓ Matrix size {n} = {grid_size}³ (consistent with 3D grid)")
        
        # Check typical 27-point stencil pattern for interior points
        # For a 3D Laplacian with 27-point stencil, each interior point connects to 27 neighbors
        max_nnz_per_row = 27
        nnz_per_row = np.count_nonzero(A, axis=1)
        
        # Count rows with different numbers of non-zeros
        boundary_rows = np.sum(nnz_per_row < max_nnz_per_row)
        interior_rows = np.sum(nnz_per_row == max_nnz_per_row)
        
        print(f"  - Interior points (27 non-zeros): {interior_rows}")
        print(f"  - Boundary points (<27 non-zeros): {boundary_rows}")
        print(f"  - Average non-zeros per row: {np.mean(nnz_per_row):.1f}")
        
        # Check if diagonal is dominant (expected for 27-point stencil)
        center_weight = np.abs(diag[0])
        print(f"  - Typical center coefficient: {center_weight:.2e}")
    else:
        print(f"\n✗ Matrix size {n} does not match a cubic grid (nearest: {grid_size}³ = {grid_size**3})")


In [9]:
def load_matrix_from_binary_check_spd(file_path: str) -> None:
    """Load a square matrix from a binary file with the specified format.
       the function to check the dense matrix properties: symmetric, SPD, condition number, diagonal dominance.
    """
    with open("../data/matrix.bin", "rb") as f:
        n_rows = np.fromfile(f, dtype=np.int32, count=1)[0]
        n_cols = np.fromfile(f, dtype=np.int32, count=1)[0]
        assert n_rows == n_cols, f"Not square: {n_rows}x{n_cols}"
        A = np.fromfile(f, dtype=np.float64, count=n_rows * n_cols).reshape(n_rows, n_cols)

    check_spd_properties(A)

In [ ]:
DATA_PATH = "../output/"

matrix_bin_path = DATA_PATH + "matrix_dense.bin"
matrix_mtx_path = DATA_PATH + "csr_matrix_full.mtx"

In [ ]:
# check the properties of the matrix loaded from the .bin file
load_matrix_from_binary_check_spd(matrix_bin_path)

In [ ]:
def load_matrix_from_mtx_and_check_spd(file_path: str) -> None:
    """Load a square matrix from a Matrix Market (.mtx) file int a compressed format."""
    from scipy.io import mmread
    A = mmread(file_path).toarray()
    n_rows, n_cols = A.shape
    assert n_rows == n_cols, f"Not square: {n_rows}x{n_cols}"
    
    check_spd_properties(A)

In [ ]:
# check the properties of the matrix loaded from the .mtx file
load_matrix_from_mtx_and_check_spd(matrix_mtx_path)

In [ ]:
def compute_speedups(serial_time: list, parallel_times: list) -> list:
    """
    Compute speedups given serial and parallel execution times.
    Parameters:
        serial_time : Execution time of the serial implementation.
        parallel_times : Execution times of the parallel implementations.
    Returns:
        List of speedup values.
    """
    return [serial_time / t for t in parallel_times]

def compute_efficiencies(speedups: list, num_processors: list) -> list:
    """
    Compute efficiencies given speedups and number of processors.
    Parameters:
        speedups : List of speedup values.
        num_processors : List of number of processors used.
    Returns:
        List of efficiency values.
    """
    return [s / p for s, p in zip(speedups, num_processors)]

In [ ]:
def read_file_to_csv(file_path: str = "../output/jcgtimes.txt") -> pd.DataFrame:
    """Read the jcgtimes.txt file into a pandas DataFrame."""
    times = pd.read_csv(file_path, sep=r'\s+')
    times_final = times.groupby(by=['#grid_size', 'num_processes'], as_index=False).agg({
        'max_time_seconds': 'mean',
        'min_time_seconds': 'mean',
    }).rename(columns={
        'max_time_seconds': 'avg_max_time_seconds',
        'min_time_seconds': 'avg_min_time_seconds'
    })
    return times_final

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Use a professional style
plt.style.use('seaborn-v0_8-paper') # or 'bmh' if seaborn is not installed
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "savefig.dpi": 300
})

def plot_report_quality(file_path="../output/jcgtimes.txt", grid_size=400):
    # Load and process data
    df = pd.read_csv(file_path)
    subset = df[df["grid_size"] == grid_size]
    
    # Average repetitions
    avg_df = subset.groupby(["algorithm", "num_processes"]).mean().reset_index()
    
    # T1 Reference: Baseline at 1 rank
    t_serial = avg_df[(avg_df["algorithm"] == "baseline") & (avg_df["num_processes"] == 1)]["total_time_max"].values[0]

    # Create figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    colors = {'baseline': '#d62728', 'pipelined': '#1f77b4'}
    markers = {'baseline': 'X', 'pipelined': 'o'}

    for alg in ['baseline', 'pipelined']:
        alg_data = avg_df[avg_df["algorithm"] == alg].sort_values("num_processes")
        procs = alg_data["num_processes"].values
        times = alg_data["total_time_max"].values
        
        speedup = t_serial / times
        efficiency = speedup / procs
        
        # --- Ax1: Speedup ---
        ax1.plot(procs, speedup, color=colors[alg], marker=markers[alg], 
                 label=f'PCG {alg.capitalize()}', linewidth=1.5, markersize=7)
        
        # --- Ax2: Efficiency ---
        ax2.plot(procs, efficiency, color=colors[alg], marker=markers[alg], 
                 label=f'PCG {alg.capitalize()}', linewidth=1.5, markersize=7)

    # --- Formatting Speedup (Ax1) ---
    max_p = avg_df["num_processes"].max()
    ax1.plot([1, max_p], [1, max_p], 'k--', alpha=0.6, label='Ideal')
    ax1.set_title(f'Strong Scaling Speedup ($Grid={grid_size}$)')
    ax1.set_xlabel('Number of Processes')
    ax1.set_ylabel('Speedup ($T_1 / T_P$)')
    ax1.set_xticks([1, 16, 32, 48, 64, 80, 96])
    ax1.grid(True, linestyle=':', alpha=0.7)
    ax1.legend()

    # --- Formatting Efficiency (Ax2) ---
    ax2.axhline(y=1.0, color='k', linestyle='--', alpha=0.6, label='Ideal')
    ax2.set_title(f'Parallel Efficiency ($Grid={grid_size}$)')
    ax2.set_xlabel('Number of Processes')
    ax2.set_ylabel('Efficiency ($S_P / P$)')
    ax2.set_xticks([1, 16, 32, 48, 64, 80, 96])
    ax2.set_ylim(0, 1.1)
    ax2.grid(True, linestyle=':', alpha=0.7)
    ax2.legend()

    # --- Annotations for Analysis ---
    # Annotate the baseline collapse
    ax1.annotate('Baseline Collapse', xy=(96, t_serial/avg_df[(avg_df["algorithm"] == "baseline") & (avg_df["num_processes"] == 96)]["total_time_max"].values[0]), 
                 xytext=(50, 40), arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5))

    # Annotate the memory wall area
    ax2.axvspan(32, 48, color='gray', alpha=0.5, zorder=0)
    ax2.text(35, 0.1, 'Memory Bandwidth Saturation', rotation=90, color='gray', fontsize=9, zorder=1)

    plt.tight_layout()
    plt.savefig(f"../plots/strong_scaling_results_grid{grid_size}.pdf")
    plt.show()

# Run the plotter
plot_report_quality()

IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def plot_internode_scaling(file_path="output/jcgtimes.txt"):
    df = pd.read_csv(file_path)
    # Average repetitions
    avg_df = df.groupby(["algorithm", "grid_size", "num_processes"]).mean().reset_index()
    
    grids = sorted(avg_df["grid_size"].unique())
    algorithms = ['baseline', 'pipelined']
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Colors for different grid sizes
    colors_512 = {'baseline': '#ff9999', 'pipelined': '#9999ff'}
    colors_875 = {'baseline': '#ff4d4d', 'pipelined': '#4d4dff'}
    colors_1100 = {'baseline': '#b30000', 'pipelined': '#0000b3'}
    
    color_map = {512: colors_512, 875: colors_875, 1100: colors_1100}
    markers = {512: 'o', 875: 's', 1100: '^'}

    for grid in grids:
        for alg in algorithms:
            data = avg_df[(avg_df["grid_size"] == grid) & (avg_df["algorithm"] == alg)].sort_values("num_processes")
            if data.empty: continue
            
            procs = data["num_processes"].values
            times = data["total_time_max"].values
            
            # Reference: Using 96 ranks as the baseline for internode scaling
            # S = (T_96 * 96) / T_P
            t_ref = data[data["num_processes"] == 96]["total_time_max"].values[0]
            speedup = (t_ref * 96) / times
            efficiency = speedup / procs
            
            label = f"{alg.capitalize()} (N={grid})"
            
            # Ax1: Relative Speedup
            ax1.plot(procs, speedup, color=color_map[grid][alg], marker=markers[grid], 
                     label=label, linewidth=2, markersize=8)
            
            # Ax2: Parallel Efficiency
            ax2.plot(procs, efficiency, color=color_map[grid][alg], marker=markers[grid], 
                     label=label, linewidth=2, markersize=8)

    # --- Formatting Ax1 (Speedup) ---
    max_p = avg_df["num_processes"].max()
    ax1.plot([96, max_p], [96, max_p], 'k--', alpha=0.6, label='Ideal')
    ax1.set_title('Internode Strong Scaling: Speedup', fontsize=14)
    ax1.set_xlabel('Total MPI Ranks', fontsize=12)
    ax1.set_ylabel('Relative Speedup (Normalized to P=96)', fontsize=12)
    ax1.set_xticks([96, 192, 384])
    ax1.grid(True, which="both", ls="-", alpha=0.2)
    ax1.legend(prop={'size': 9})

    # --- Formatting Ax2 (Efficiency) ---
    ax2.axhline(y=1.0, color='k', linestyle='--', alpha=0.6)
    ax2.set_title('Internode Parallel Efficiency', fontsize=14)
    ax2.set_xlabel('Total MPI Ranks', fontsize=12)
    ax2.set_ylabel('Efficiency ($E = S_p / P$)', fontsize=12)
    ax2.set_xticks([96, 192, 384])
    ax2.set_ylim(0, 1.1)
    ax2.grid(True, which="both", ls="-", alpha=0.2)
    
    # Add a text box highlighting the Baseline failure
    ax1.text(200, 120, "Baseline overhead dominated\nby global Allgatherv", 
             bbox=dict(facecolor='white', alpha=0.5), fontsize=10)

    plt.tight_layout()
    plt.savefig("internode_scaling_comparison.pdf")
    plt.show()

# Execute
plot_internode_scaling()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

def plot_convergence_analysis(baseline_file: str = "../output/residual_baseline.txt",
                              pipelined_file: str = "../output/residual_pipelined.txt",
                              output_file_path: str = "../plots/convergence_analysis.eps") -> None:
    """
    Plot convergence analysis for the research report.
    Assumes files have columns: #iteration  residual_norm
    """
    
    # 1. Load Data (skiprows/comment handles the '#' in your C output)
    try:
        baseline_data = pd.read_csv(baseline_file, sep=r'\s+', comment='#', names=['iteration', 'residual'])
        pipelined_data = pd.read_csv(pipelined_file, sep=r'\s+', comment='#', names=['iteration', 'residual'])
    except FileNotFoundError as e:
        print(f"Error: Could not find result files. {e}")
        return

    # 2. Setup Plot Style
    plt.style.use('seaborn-v0_8-paper') # Or 'ggplot'
    plt.figure(figsize=(8, 5))

    # 3. Plot trajectories
    # We use markevery=10 so markers don't overlap too much
    plt.semilogy(baseline_data['iteration'], baseline_data['residual'], 
                 label='Baseline Jacobi-CG', 
                 color='blue', linestyle='-', marker='o', 
                 markevery=10, markersize=5, linewidth=1.5)

    plt.semilogy(pipelined_data['iteration'], pipelined_data['residual'], 
                 label='Pipelined Jacobi-CG', 
                 color='red', linestyle='--', marker='s', 
                 markevery=10, markersize=5, linewidth=1.5)

    # 4. Add Reference Tolerance Line (10^-10)
    plt.axhline(y=1e-10, color='gray', linestyle=':', alpha=0.7, label='Target Tolerance ($10^{-10}$)')

    # 5. Labels and Titles (Standard for academic papers)
    plt.title('Residual Convergence: Baseline vs. Pipelined JCG', fontsize=14)
    plt.xlabel('Iteration Number ($k$)', fontsize=12)
    plt.ylabel(r'Relative Residual $\frac{||r_k||_2}{||r_0||_2}$', fontsize=12)
    
    plt.legend(loc='upper right', frameon=True)
    plt.grid(True, which="both", ls="-", alpha=0.2)
    
    # 6. Save and clean up
    # Ensure plots directory exists
    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
    
    plt.tight_layout()
    plt.savefig(output_file_path, format='eps', dpi=300)
    print(f"Plot saved to: {output_file_path}")
    plt.show()

In [ ]:
plot_convergence_analysis()